In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------
N_SPLITS = 5
N_PERM   = 200
C        = 1.0
RANDOM_STATE = 0

VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'
AREAS_PATH  = '/home/maria/ProjectionSort/data/brain_area.npy'

vit   = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R     = np.load(NEURAL_PATH).T                                   # (images, neurons)  <-- should be in (0,1)
areas = np.load(AREAS_PATH, allow_pickle=True)

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# Animate / inanimate label (your heuristic)
top1 = np.argmax(vit, axis=1)
y_true = (top1 <= 397).astype(int)
print("Animate fraction:", y_true.mean())

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

X = StandardScaler().fit_transform(R)
y = y_true

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

accs = []
sv_fracs = []
margins = []

for train, test in cv.split(X, y):
    clf = LinearSVC(C=C, dual=False, max_iter=5000)
    clf.fit(X[train], y[train])

    yhat = clf.predict(X[test])
    accs.append(accuracy_score(y[test], yhat))

    # margin = 1 / ||w||
    w = clf.coef_.ravel()
    margins.append(1.0 / np.linalg.norm(w))

    # which points are "support-like": |f(x)| <= 1
    f = X[train] @ w + clf.intercept_[0]
    sv_frac = np.mean(np.abs(f) <= 1.0)
    sv_fracs.append(sv_frac)

print("Linear SVM accuracy:", np.mean(accs))
print("Mean margin:", np.mean(margins))
print("Support-vector fraction:", np.mean(sv_fracs))


Images: 118
Neurons: 39209
Animate fraction: 0.5338983050847458
Linear SVM accuracy: 0.6782608695652174
Mean margin: 19.901305678940773
Support-vector fraction: 0.27088465845464726


In [2]:
from sklearn.svm import SVC

accs_lin = []
accs_rbf = []

for train, test in cv.split(X, y):

    lin = LinearSVC(C=C, dual=False, max_iter=5000)
    rbf = SVC(C=1.0, kernel='rbf', gamma='scale')

    lin.fit(X[train], y[train])
    rbf.fit(X[train], y[train])

    accs_lin.append(accuracy_score(y[test], lin.predict(X[test])))
    accs_rbf.append(accuracy_score(y[test], rbf.predict(X[test])))

print("Linear SVM:", np.mean(accs_lin))
print("RBF SVM:", np.mean(accs_rbf))


Linear SVM: 0.6782608695652174
RBF SVM: 0.6768115942028985


In [3]:
clf = LinearSVC(C=C, dual=False, max_iter=5000).fit(X, y)

w = clf.coef_.ravel()
f = X @ w + clf.intercept_[0]
sv_idx = np.where(np.abs(f) <= 1.0)[0]

print("Number of boundary images:", len(sv_idx))
print("Animate fraction among SVs:", y[sv_idx].mean())


Number of boundary images: 51
Animate fraction among SVs: 1.0
